# Phase 2 — Ensemble Blender (RSNA Knee Abnormality Detection)

Blends three prediction sources into one submission:

1. **0.954 fork** (`submission.csv` from heliosli's blend) — weight 0.6
2. **ConvNeXt-Tiny** (`convnext_best.pt`, your trained model) — weight 0.2
3. **CORAL** (`coral_best.pt`, your trained model, binary heads only) — weight 0.2

Blending is done in **rank space** (per-column ranks, weighted average) — the
metric reads only order, so ranks discard nothing.

**Inference only — no training.** Both models run on test DICOMs with the same
preprocessing as training (288px, 8 slices/plane, no augmentation).



In [ ]:
# ================= FAIL-FAST + CONFIG =================
VERSION = "phase2-ensemble-blend-v1 (2026-10-12: blends 0.954 fork + ConvNeXt-Tiny + CORAL)"
print(f"VERSION: {VERSION}")
print("EXPECTED FIRST OUTPUT: the VERSION line above. If it differs, delete this copy (stale).")

import os
from pathlib import Path

PROBE = True            # True: 5 studies to validate pipeline. False: full test run.
PROBE_N = 5

# --- blend weights (must sum to 1) ---
W_FORK = 0.6
W_CONVNEXT = 0.2
W_CORAL = 0.2
assert abs(W_FORK + W_CONVNEXT + W_CORAL - 1.0) < 1e-9, "weights must sum to 1"

# --- shared constants (must match training) ---
IMAGE_SIZE = 288
N_SLICES = 8
PLANES = ["axial", "coronal", "sagittal"]
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
N_FINDINGS = 12
SUB_COLS = ["StudyInstanceUID"] + LABELS
ATTN_DIM = 128
SEED = 42
OUT_CSV = "/kaggle/working/submission.csv"

# --- fail-fast: all three sources must exist ---
def _first_existing(cands):
    for p in cands:
        if os.path.exists(p):
            return p
    return None

# 1) 0.954 fork submission.csv
FORK_CANDIDATES = [
    "/kaggle/working/submission.csv",                       # run the fork notebook first, in this session
    "/kaggle/input/fork-submission/submission.csv",          # or attach it as a dataset
]
for _root, _dirs, _files in os.walk("/kaggle/input"):
    if "submission.csv" in _files and "fork" in _root.lower():
        FORK_CANDIDATES.append(os.path.join(_root, "submission.csv"))
        break
FORK_CSV = _first_existing(FORK_CANDIDATES)

# 2) ConvNeXt checkpoint
CN_CANDIDATES = [
    "/kaggle/input/datasets/sridharshivashankar/convnext-ckpt/convnext_best.pt",
    "/kaggle/working/convnext_ckpt/convnext_best.pt",
]
for _root, _dirs, _files in os.walk("/kaggle/input"):
    if "convnext_best.pt" in _files:
        CN_CANDIDATES.append(os.path.join(_root, "convnext_best.pt"))
        break
CN_CKPT = _first_existing(CN_CANDIDATES)

# 3) CORAL checkpoint
CORAL_CANDIDATES = [
    "/kaggle/input/datasets/sridharshivashankar/coral-ckpt/coral_best.pt",
    "/kaggle/working/coral_ckpt/coral_best.pt",
]
for _root, _dirs, _files in os.walk("/kaggle/input"):
    if "coral_best.pt" in _files:
        CORAL_CANDIDATES.append(os.path.join(_root, "coral_best.pt"))
        break
CORAL_CKPT = _first_existing(CORAL_CANDIDATES)

# 4) competition data (test DICOMs)
COMP_DATA = None
for _cand in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
              "/kaggle/input/rsna-knee-abnormality-detection"]:
    if os.path.isdir(_cand):
        COMP_DATA = _cand
        break

print(f"  FORK_CSV   = {FORK_CSV}")
print(f"  CN_CKPT    = {CN_CKPT}")
print(f"  CORAL_CKPT = {CORAL_CKPT}")
print(f"  COMP_DATA  = {COMP_DATA}")

_missing = []
if FORK_CSV is None:
    _missing.append("0.954 fork submission.csv (run the fork notebook first, or attach its submission.csv)")
if CN_CKPT is None:
    _missing.append("convnext_best.pt (train ConvNeXt-Tiny first, or attach the checkpoint)")
if CORAL_CKPT is None:
    _missing.append("coral_best.pt (train CORAL first, or attach the checkpoint)")
if COMP_DATA is None:
    _missing.append("competition data (attach rsna-knee-abnormality-detection)")
if _missing:
    raise RuntimeError("FATAL [blend]: missing inputs:\n  - " + "\n  - ".join(_missing))

# --- fail-fast: fork csv columns must match exactly ---
import pandas as _pd
_fork_head = _pd.read_csv(FORK_CSV, nrows=3)
print(f"  fork csv shape sample: {_fork_head.shape}, cols: {list(_fork_head.columns)[:4]}...")
if list(_fork_head.columns) != SUB_COLS:
    raise RuntimeError(
        f"FATAL [blend]: fork submission.csv columns do not match.\n"
        f"  expected: {SUB_COLS}\n  got: {list(_fork_head.columns)}")
print("  fork csv columns OK")
del _pd, _fork_head

print("\nCONFIG DUMP")
for k in ['PROBE','PROBE_N','W_FORK','W_CONVNEXT','W_CORAL','IMAGE_SIZE','N_SLICES','OUT_CSV']:
    print(f"{k} = {globals()[k]}")
print("fail-fast: ALL GREEN")



In [ ]:
# ================= SETUP =================
import time, math, re, hashlib, warnings
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom as _zoom

t_start = time.time()
torch.manual_seed(SEED); np.random.seed(SEED)

# Single GPU only — DataParallel crashes CUDA kernels on T4 (proven twice).
assert torch.cuda.is_available(), "FATAL: no GPU"
DEVICE = torch.device("cuda:0")
torch.cuda.set_device(0)
print(f"torch {torch.__version__} | device: {torch.cuda.get_device_name(0)}")

# ---- discover test studies ----
TEST_DIR_CANDIDATES = ([f"{COMP_DATA}/test_series",
                        f"{COMP_DATA}/test",
                        f"{COMP_DATA}/test_images"] if COMP_DATA else [])
test_dir = next((d for d in TEST_DIR_CANDIDATES if os.path.isdir(d)), None)
print(f"test dir: {test_dir}")
assert test_dir, f"FATAL: no test dir found in {TEST_DIR_CANDIDATES}"

# sample submission defines the expected study list + column order
sample_sub = None
for cand in [f"{COMP_DATA}/sample_submission.csv",
             f"{COMP_DATA}/sampleSubmission.csv",
             f"{COMP_DATA}/sample-submission.csv"]:
    if os.path.exists(cand):
        sample_sub = pd.read_csv(cand)
        print(f"  using sample submission: {cand}")
        break
assert sample_sub is not None, "FATAL: sample_submission.csv not found"
assert list(sample_sub.columns) == SUB_COLS, f"sample_submission columns mismatch: {list(sample_sub.columns)}"
test_uids = sample_sub["StudyInstanceUID"].tolist()
print(f"  {len(test_uids)} studies from sample_submission.csv")

# cross-check against test dir
_dir_uids = sorted([d.name for d in Path(test_dir).iterdir() if d.is_dir()])
_missing_dirs = [u for u in test_uids if u not in set(_dir_uids)]
print(f"  test_dir has {len(_dir_uids)} study dirs | uids w/o dir: {len(_missing_dirs)}")
if _missing_dirs:
    print(f"  first missing: {_missing_dirs[:5]} (will 0.5-fill)")

if PROBE:
    test_uids = test_uids[:PROBE_N]
    print(f"  PROBE: {len(test_uids)} studies only")

# ---- fork predictions (aligned to test_uids order) ----
fork_df = pd.read_csv(FORK_CSV)
assert list(fork_df.columns) == SUB_COLS, "fork csv columns changed since fail-fast!"
fork_df = fork_df.set_index("StudyInstanceUID")
_missing_fork = [u for u in test_uids if u not in fork_df.index]
if _missing_fork:
    raise RuntimeError(f"FATAL [blend]: {len(_missing_fork)} test uids missing from fork csv "
                       f"(first: {_missing_fork[:3]}). The fork must cover the full test set.")
fork_probs = fork_df.loc[test_uids, LABELS].to_numpy(dtype=np.float64)
assert fork_probs.shape == (len(test_uids), N_FINDINGS)
assert np.all(np.isfinite(fork_probs)), "NaNs in fork predictions"
print(f"  fork predictions: {fork_probs.shape}, range [{fork_probs.min():.4f}, {fork_probs.max():.4f}]")
print("setup: ALL GREEN")



In [ ]:
# ================= PHASE-0 LITE (test DICOM -> 336px volumes) =================
# Same deterministic pipeline as training cache. In-memory, no sharding.
# Uses scipy for resampling (no cv2 dependency).

FATSAT_RE = re.compile(r"fat[\s\-_]*sat")

def decode_slice(path):
    ds = pydicom.dcmread(str(path))
    x = ds.pixel_array.astype(np.float32)
    x = x * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        x = x.max() - x
    return x, ds

def geometric_order(datasets):
    keyed = []
    for ds in datasets:
        try:
            iop = [float(v) for v in ds.ImageOrientationPatient]
            ipp = [float(v) for v in ds.ImagePositionPatient]
            if len(iop) != 6 or len(ipp) != 3:
                raise ValueError("bad IOP/IPP")
            row = np.array(iop[0:3]); col = np.array(iop[3:6])
            normal = np.cross(row, col)
            n = float(np.linalg.norm(normal))
            if n < 1e-6:
                raise ValueError("degenerate IOP")
            normal = normal / n
            keyed.append((float(np.dot(np.array(ipp), normal)), ds))
        except Exception:
            keyed = None
            break
    if keyed is None:
        def _inum(ds):
            try: return int(ds.InstanceNumber)
            except Exception: return 0
        ordered = sorted(datasets, key=lambda ds: (_inum(ds), str(getattr(ds, "SOPInstanceUID", ""))))
        return ordered, None, True
    keyed.sort(key=lambda kv: kv[0])
    ds0 = keyed[0][1]
    iop = [float(v) for v in ds0.ImageOrientationPatient]
    normal = np.cross(np.array(iop[0:3]), np.array(iop[3:6]))
    normal = normal / float(np.linalg.norm(normal))
    return [ds for _, ds in keyed], normal, False

def window_series(vol, pct_lo=1.0, pct_hi=99.0):
    lo, hi = np.percentile(vol, [pct_lo, pct_hi])
    if not (np.isfinite(lo) and np.isfinite(hi)) or hi <= lo:
        return np.zeros_like(vol), False
    vol = np.clip(vol, lo, hi)
    return (vol - lo) / (hi - lo), True

def resample_slice(img, orig_spacing, target_spacing):
    sh, sw = img.shape
    nh = max(1, int(round(sh * orig_spacing[0] / target_spacing)))
    nw = max(1, int(round(sw * orig_spacing[1] / target_spacing)))
    if (nh, nw) == (sh, sw):
        return img
    return _zoom(img, (nh / sh, nw / sw), order=1).astype(np.float32)

def center_crop_pad(img, out_px, pad_value=0):
    h, w = img.shape
    ph = max(0, out_px - h); pw = max(0, out_px - w)
    if ph or pw:
        img = np.pad(img, ((ph // 2, ph - ph // 2), (pw // 2, pw - pw // 2)),
                     mode="constant", constant_values=pad_value)
        h, w = img.shape
    sh = (h - out_px) // 2; sw = (w - out_px) // 2
    return img[sh:sh + out_px, sw:sw + out_px]

def plane_from_iop(iop):
    row = np.array(iop[0:3]); col = np.array(iop[3:6])
    normal = np.cross(row, col)
    n = float(np.linalg.norm(normal))
    if n < 1e-6:
        return "unknown"
    normal = normal / n
    ax = int(np.argmax(np.abs(normal)))
    if abs(normal[ax]) < 0.8:
        return "unknown"
    return ["sagittal", "coronal", "axial"][ax]

def is_fluid_sensitive(ds):
    desc = str(getattr(ds, "SeriesDescription", "") or "")
    seq = str(getattr(ds, "ScanningSequence", "") or "")
    try: te = float(getattr(ds, "EchoTime", 0) or 0)
    except Exception: te = 0.0
    try: tr = float(getattr(ds, "RepetitionTime", 0) or 0)
    except Exception: tr = 0.0
    dl = desc.lower()
    toks = set(t for t in re.split(r"[^a-z0-9]+", dl) if t)
    seq_vals = [s.strip().upper() for s in seq.split("\\") if s.strip()]
    if "GR" in seq_vals:
        return False, "seq:GRE"
    fs_hit = sorted(toks & {"fs", "stir", "pd", "t2"})
    if FATSAT_RE.search(dl):
        fs_hit.append("fatsat")
    fs_hit.sort()
    if fs_hit:
        return True, "desc:" + "+".join(fs_hit)
    if "t1" in toks:
        return False, "desc:t1"
    if te >= 40:
        return True, "trte:TE>=40"
    if 0 < te < 20 and 0 < tr < 800:
        return False, "trte:T1-like"
    return False, "unknown->not-FS"

def canonicalize_laterality(vol, iop, laterality, normal):
    """Mirror R knees to canonical left; canonicalize sagittal stack order.
    (Full Phase-0 logic — training cache was canonicalized, so inference must be too.)"""
    X = np.array([1.0, 0.0, 0.0])
    if normal is not None:
        nx = abs(float(np.dot(normal, X)))
    else:
        nx = 0.0
    if nx > 0.8:
        # sagittal: force lateral->medial stack order for both L and R
        # lateral->medial runs along -X for L, +X for R
        if laterality == "R":
            return vol[::-1], True, "sagittal R: reversed to lateral->medial"
        return vol, False, "sagittal L/U: kept"
    # coronal/axial: mirror R only
    if laterality == "R":
        return vol[:, :, ::-1], True, "R mirrored to canonical left"
    return vol, False, "L/U: no-op"

def process_series(series_dir):
    files = sorted([f for f in Path(series_dir).iterdir()
                    if f.is_file() and f.suffix.lower() == ".dcm"])
    if not files:
        raise ValueError("no DICOM files")
    pairs = []
    for f in files:
        x, ds = decode_slice(f)
        pairs.append((ds, x))
    dss = [ds for ds, _ in pairs]
    ordered, normal, _fb = geometric_order(dss)
    arr_by_id = {id(ds): x for ds, x in pairs}
    vol = np.stack([arr_by_id[id(ds)] for ds in ordered]).astype(np.float32)
    if not np.all(np.isfinite(vol)):
        raise ValueError("non-finite pixels")
    vol, win_ok = window_series(vol, P0_PCT_LO, P0_PCT_HI)
    ds0 = ordered[0]
    try:
        ps = ds0.PixelSpacing
        spacing = (float(ps[0]), float(ps[1]))
    except Exception:
        raise ValueError("PixelSpacing missing")
    out = np.stack([resample_slice(s, spacing, P0_SPACING_MM) for s in vol])
    out = np.stack([center_crop_pad(s, P0_TARGET_PX, 0) for s in out])
    assert out.shape[1] == P0_TARGET_PX and out.shape[2] == P0_TARGET_PX
    try:
        iop = [float(v) for v in ds0.ImageOrientationPatient]
    except Exception:
        raise ValueError("ImageOrientationPatient missing")
    plane_tag = str(getattr(ds0, "Anatomical_Plane", "") or "").strip().lower()
    plane = plane_tag if plane_tag in ("sagittal", "coronal", "axial") else plane_from_iop(iop)
    fs, fs_reason = is_fluid_sensitive(ds0)
    meta = {
        "series_uid": str(getattr(ds0, "SeriesInstanceUID", Path(series_dir).name)),
        "plane": plane, "is_fs": bool(fs), "fs_reason": fs_reason,
        "n_slices": int(out.shape[0]),
        "laterality": str(getattr(ds0, "ImageLaterality", "") or "").strip().upper(),
        "iop": iop, "normal": normal,
    }
    return out, meta

def select_series(series_infos):
    by_plane_fs, by_plane_all = {}, {}
    for vol, meta in series_infos:
        if meta["plane"] not in ("sagittal", "coronal", "axial"):
            continue
        by_plane_all.setdefault(meta["plane"], []).append((vol, meta))
        if meta["is_fs"]:
            by_plane_fs.setdefault(meta["plane"], []).append((vol, meta))
    selected = {}
    for plane in ("sagittal", "coronal", "axial"):
        cands = by_plane_fs.get(plane, [])
        if not cands:
            cands = by_plane_all.get(plane, [])
        if not cands:
            continue
        cands.sort(key=lambda vm: (-vm[1]["n_slices"], vm[1]["series_uid"]))
        selected[plane] = cands[0]
    return selected

def preprocess_study(study_dir):
    _V = True  # verbose per-study preprocessing
    if _V: print(f"    [preprocess] dir: {study_dir}")
    """Test DICOM study -> {plane: uint8 (S,336,336)}, plane_mask. Never crashes caller."""
    series_dirs = sorted([d for d in Path(study_dir).iterdir() if d.is_dir()])
    if not series_dirs:
        raise ValueError("no series directories")
    infos = []
    for sdir in series_dirs:
        try:
            infos.append(process_series(sdir))
        except Exception:
            continue
    if not infos:
        raise ValueError("all series failed")
    lats = [m["laterality"] for _, m in infos if m["laterality"] in ("L", "R")]
    laterality = max(sorted(set(lats)), key=lats.count) if lats else "U"
    selected = select_series(infos)
    if not selected:
        raise ValueError("no usable plane")
    planes_out, mask = {}, []
    for p in PLANES:
        if p in selected:
            vol, meta = selected[p]
            normal = None if meta["normal"] is None else np.array(meta["normal"])
            vol_m, _, _ = canonicalize_laterality(vol, meta["iop"], laterality, normal)
            u8 = np.clip(vol_m, 0.0, 1.0)
            u8 = (u8 * 255.0 + 0.5).astype(np.uint8)
            planes_out[p] = u8
            mask.append(1.0)
        else:
            mask.append(0.0)
    return planes_out, mask

print("phase-0 lite: loaded")



In [ ]:
import timm

class GatedAttentionMIL(nn.Module):
    def __init__(self, feat_dim, attn_dim=ATTN_DIM):
        super().__init__()
        self.V = nn.Linear(feat_dim, attn_dim)
        self.U = nn.Linear(feat_dim, attn_dim)
        self.w = nn.Linear(attn_dim, 1)

    def forward(self, h):  # h: (B, S, D)
        a = self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h)))
        a = F.softmax(a, dim=1)
        return (a * h).sum(dim=1), a.squeeze(-1)

class KneeConvNeXtMIL(nn.Module):
    def __init__(self, backbone_name="convnext_tiny", attn_dim=ATTN_DIM,
                 n_findings=N_FINDINGS, pretrained=True,
                 drop_path_rate=0.1):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, drop_path_rate=drop_path_rate)
        print(f'backbone init OK: {backbone_name} pretrained={pretrained}')
        D = self.backbone.num_features
        print(f'backbone feature dim: {D}')
        self.mil_heads = nn.ModuleList([GatedAttentionMIL(D, attn_dim)
                                        for _ in range(n_findings)])
        self.heads = nn.ModuleList([nn.Linear(D, 1) for _ in range(n_findings)])

    def forward_plane(self, x):  # x: (B, S, 3, H, W)
        B, S = x.shape[:2]
        # reshape (not view): non-contiguous tensors crash view() on CUDA
        h = self.backbone(x.reshape(B * S, *x.shape[2:])).reshape(B, S, -1)
        return torch.stack([mil(h)[0] for mil in self.mil_heads], dim=1)  # (B,12,D)

    def forward(self, planes, plane_mask):
        feats = torch.stack([self.forward_plane(planes[p]) for p in PLANES], dim=1)
        m = plane_mask.reshape(-1, 3, 1, 1)
        fused = (feats * m).sum(1) / m.sum(1).clamp(min=1e-6)  # (B,12,D)
        return torch.cat([self.heads[f](fused[:, f, :])
                          for f in range(N_FINDINGS)], dim=1)  # (B,12)

    @property
    def enc(self):
        return self.backbone  # for frozen-BN helper



class KneeCoralMIL(nn.Module):
    def __init__(self, backbone_name="efficientnet_b3", attn_dim=ATTN_DIM,
                 n_findings=N_FINDINGS, pretrained=True,
                 drop_path_rate=0.1):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, drop_path_rate=drop_path_rate)
        print(f'backbone init OK: {backbone_name} pretrained={pretrained}')
        D = self.backbone.num_features
        print(f'backbone feature dim: {D}')
        self.mil_heads = nn.ModuleList([GatedAttentionMIL(D, attn_dim)
                                        for _ in range(n_findings)])
        self.heads = nn.ModuleList([nn.Linear(D, 1) for _ in range(n_findings)])
        # CORAL ordinal heads: for each severity finding, 3 threshold classifiers
        # P(y>0), P(y>1), P(y>2) on that finding's fused MIL feature.
        self.coral_heads = nn.ModuleList([
            nn.ModuleList([nn.Linear(D, 1) for _ in range(N_CORAL_THRESHOLDS)])
            for _ in CORAL_FINDING_IDX
        ])

    def forward_plane(self, x):  # x: (B, S, 3, H, W)
        B, S = x.shape[:2]
        # reshape (not view): non-contiguous tensors crash view() on CUDA
        h = self.backbone(x.reshape(B * S, *x.shape[2:])).reshape(B, S, -1)
        return torch.stack([mil(h)[0] for mil in self.mil_heads], dim=1)  # (B,12,D)

    def forward(self, planes, plane_mask):
        feats = torch.stack([self.forward_plane(planes[p]) for p in PLANES], dim=1)
        m = plane_mask.reshape(-1, 3, 1, 1)
        fused = (feats * m).sum(1) / m.sum(1).clamp(min=1e-6)  # (B,12,D)
        binary_logits = torch.cat([self.heads[f](fused[:, f, :])
                                   for f in range(N_FINDINGS)], dim=1)  # (B,12)
        # CORAL: (B,4,3) — thresholds for each severity finding
        coral_logits = torch.stack([
            torch.cat([thr(fused[:, fi, :]) for thr in thr_heads], dim=1)
            for fi, thr_heads in zip(CORAL_FINDING_IDX, self.coral_heads)
        ], dim=1)  # (B,4,3)
        return binary_logits, coral_logits

    @property
    def enc(self):
        return self.backbone  # for frozen-BN helper



# CORAL constants (must match training cell 0)
CORAL_FINDING_IDX = [8, 6, 4, 5]   # Synovitis, PF OA, Medial OA, Lateral OA
N_CORAL_THRESHOLDS = 3

print("model classes defined: KneeConvNeXtMIL, KneeCoralMIL")



In [ ]:
# ================= INFERENCE (both models) =================
def sample_slices(vol, n):
    S = vol.shape[0]
    if S == n:
        return vol
    idx = np.linspace(0, S - 1, n).round().astype(int)
    return vol[idx]

def study_to_batch(planes_u8, plane_mask):
    """{plane: (S,336,336) uint8} -> model-ready tensors (no augmentation)."""
    planes = {}
    for p, m in zip(PLANES, plane_mask):
        if m == 1.0:
            vol = sample_slices(planes_u8[p], N_SLICES).astype(np.float32) / 255.0
            t = torch.from_numpy(vol).unsqueeze(1)  # (S,1,336,336)
            t = F.interpolate(t, size=(IMAGE_SIZE, IMAGE_SIZE),
                              mode="bilinear", align_corners=False)
            vol = t.squeeze(1).numpy().astype(np.float32)
        else:
            vol = np.zeros((N_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        planes[p] = np.repeat(vol[:, None, :, :], 3, axis=1).astype(np.float32)
    return planes, np.array(plane_mask, dtype=np.float32)

def load_model_weights(model, ckpt_path, name):
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    print(f"  {name} checkpoint keys: {list(ck.keys())}")
    # Prefer EMA weights (best), fall back to raw model.
    state_key = "ema" if isinstance(ck.get("ema"), dict) else "model"
    tgt = model.module if isinstance(model, nn.DataParallel) else model
    incompat = tgt.load_state_dict(ck[state_key], strict=False)
    print(f"  loaded {ckpt_path} [{state_key}] "
          f"(epoch {ck.get('epoch')}, best {ck.get('best')})")
    print(f"  missing={incompat.missing_keys}, unexpected={incompat.unexpected_keys}")
    assert not incompat.missing_keys, f"{name}: missing keys on load!"
    return model

@torch.no_grad()
def predict_with_model(model, uids, test_dir, name):
    """Run inference. Returns (n, 12) float64 probs. Failed study -> 0.5s."""
    model.eval()
    results, failed = [], []
    t0 = time.time()
    for i, uid in enumerate(uids):
        try:
            sdir = os.path.join(test_dir, uid)
            if not os.path.isdir(sdir):
                raise ValueError("study dir missing")
            planes_u8, pmask = preprocess_study(sdir)
            planes_np, pmask_np = study_to_batch(planes_u8, pmask)
            batch = {p: torch.from_numpy(planes_np[p]).unsqueeze(0).to(DEVICE) for p in PLANES}
            out = model(batch, torch.from_numpy(pmask_np).unsqueeze(0).to(DEVICE))
            logits = out[0] if isinstance(out, tuple) else out  # CORAL returns (binary, coral)
            probs = torch.sigmoid(logits).double().cpu().numpy()[0]
            assert probs.shape == (N_FINDINGS,) and np.all(np.isfinite(probs))
        except Exception as e:
            probs = np.full(N_FINDINGS, 0.5)
            failed.append(uid)
            if len(failed) <= 3:
                print(f"  FAILED {uid}: {type(e).__name__}: {e}")
        results.append(probs)
        if (i + 1) % 20 == 0 or (i + 1) == len(uids):
            dt = time.time() - t0
            rate = (i + 1) / max(dt, 1e-6)
            eta = (len(uids) - i - 1) / rate if rate > 0 else 0
            print(f"  [{name}] {i+1}/{len(uids)} | {dt:.0f}s | ETA {eta/60:.0f}min | failed: {len(failed)}",
                  flush=True)
    P = np.stack(results)
    print(f"  [{name}] done: {P.shape}, failed: {len(failed)}, "
          f"range [{P.min():.4f}, {P.max():.4f}]")
    return P

print("building ConvNeXt-Tiny...")
assert "convnext_tiny" in timm.list_models(), "FATAL: convnext_tiny not in timm"
convnext_model = KneeConvNeXtMIL(backbone_name="convnext_tiny", pretrained=False,
                                 drop_path_rate=0.0).to(DEVICE)
convnext_model = load_model_weights(convnext_model, CN_CKPT, "convnext")

print("building CORAL (EfficientNet-B3 + coral heads)...")
assert "efficientnet_b3" in timm.list_models(), "FATAL: efficientnet_b3 not in timm"
coral_model = KneeCoralMIL(backbone_name="efficientnet_b3", pretrained=False,
                            drop_path_rate=0.0).to(DEVICE)
coral_model = load_model_weights(coral_model, CORAL_CKPT, "coral")

print("\nrunning ConvNeXt inference...")
cn_probs = predict_with_model(convnext_model, test_uids, test_dir, "convnext")
del convnext_model
torch.cuda.empty_cache()

print("\nrunning CORAL inference (binary heads only)...")
coral_probs = predict_with_model(coral_model, test_uids, test_dir, "coral")
del coral_model
torch.cuda.empty_cache()
print("inference: ALL GREEN")



In [ ]:
# ================= BLEND (rank space) =================
from scipy.stats import rankdata

def to_ranks(P):
    """Per-column ranks, higher pred -> higher rank. Returns float64 (n, 12)."""
    R = np.empty_like(P)
    for j in range(P.shape[1]):
        R[:, j] = rankdata(P[:, j], method="average")
    return R

print("ranking predictions per column...")
r_fork = to_ranks(fork_probs)
r_cn = to_ranks(cn_probs)
r_coral = to_ranks(coral_probs)
n = len(test_uids)
print(f"  ranks computed on {n} studies")

blended_ranks = W_FORK * r_fork + W_CONVNEXT * r_cn + W_CORAL * r_coral
# Normalize ranks to (0, 1] — order is all the metric reads.
final_probs = blended_ranks / n
print(f"  blend weights: fork={W_FORK}, convnext={W_CONVNEXT}, coral={W_CORAL}")
print(f"  final range: [{final_probs.min():.4f}, {final_probs.max():.4f}]")

rows = [[uid] + [float(p) for p in probs] for uid, probs in zip(test_uids, final_probs)]
sub = pd.DataFrame(rows, columns=SUB_COLS)
sub.to_csv(OUT_CSV, index=False)
print(f"\nwrote {OUT_CSV}  shape={sub.shape}")
print("blend: ALL GREEN")



In [ ]:
# ================= VERIFY SUBMISSION =================
sub = pd.read_csv(OUT_CSV)
print(f"shape: {sub.shape} (expect ({len(test_uids)}, 13))")
assert list(sub.columns) == SUB_COLS, f"column mismatch: {list(sub.columns)}"
assert len(sub) == len(test_uids), "row count mismatch"
assert sub["StudyInstanceUID"].tolist() == test_uids, "uid order mismatch"
assert sub[LABELS].notna().all().all(), "NaNs in predictions"
assert ((sub[LABELS] >= 0) & (sub[LABELS] <= 1)).all().all(), "probs out of [0,1]"
print("no NaNs, all probs in [0,1], uids match")
print("\nhead:")
print(sub.head(3).round(4).to_string(index=False))
print(f"\ntotal time: {(time.time()-t_start)/60:.1f} min")
print("VERIFY: ALL GREEN")

